In [1]:
# Combined Part B + Part D baseline: k-fold-by-NPI, same protocol as Notebooks 05 / 08.
# SMOKE=1 (default) = 1 run, 1 fold. sbatch sets SMOKE=0.
# LABEL_COL: fraud | fraud_h2 | fraud_nob4.  LEARNERS: "XGB" (default) or "XGB,RF".
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

SMOKE     = os.environ.get("SMOKE", "1") == "1"
LABEL_COL = os.environ.get("LABEL_COL", "fraud")
LEARNERS  = os.environ.get("LEARNERS", "XGB").split(",")
N_JOBS    = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
BD       = DATA_DIR / "processed/combined/bd_labeled_2019_2023.parquet"
RES_DIR  = DATA_DIR / "results/combined"; RES_DIR.mkdir(parents=True, exist_ok=True)
TAG      = f"nb11_{LABEL_COL}" + ("_smoke" if SMOKE else "")
RES_CSV  = RES_DIR / f"{TAG}_fold_results.csv"
print("SMOKE:", SMOKE, "| label:", LABEL_COL, "| learners:", LEARNERS, "| results →", RES_CSV.name)

SMOKE: False | label: fraud_nob4 | learners: ['XGB'] | results → nb11_fraud_nob4_fold_results.csv


## Design

Three feature sets, trained on **all** provider-years, same folds:
- **B** — Part B features + Part B provider type
- **D** — Part D features + Part D provider type
- **BD** — both

All three include `has_partb` / `has_partd`. Each test fold is scored overall and **per group** (`both`, `B_only`, `D_only`). The key comparison is on `both` rows: does seeing billing **and** prescribing beat either one alone on the same providers?

RF is off by default (slow on ~7M rows × ~400 columns); set `LEARNERS=XGB,RF` to add it.

In [2]:
N_RUNS, N_FOLDS = (1, 5) if SMOKE else (6, 5)
FOLDS_TO_RUN    = [0] if SMOKE else list(range(N_FOLDS))

def make_model(name, seed):
    if name == "XGB":
        return XGBClassifier(max_depth=4, n_jobs=N_JOBS, random_state=seed,
                             tree_method="hist", eval_metric="logloss")
    return RandomForestClassifier(max_depth=16, n_jobs=N_JOBS, random_state=seed)

In [3]:
df = pd.read_parquet(BD)
LABELS = ["fraud", "fraud_h2", "fraud_nob4"]
df = df[df[LABEL_COL] >= 0].reset_index(drop=True)          # -1 = excluded for this label
print(df.shape, "| fraud rows:", int(df[LABEL_COL].sum()))

B_FEATS = [c for c in df.columns if c.startswith("b_") and c not in ("b_type", "b_ent_cd")]
D_FEATS = [c for c in df.columns if c.startswith("d_") and c != "d_type"]
FLAGS   = ["has_partb", "has_partd"]
print("B feats:", len(B_FEATS), "| D feats:", len(D_FEATS))

b_pt = pd.get_dummies(df["b_type"].where(df.has_partb == 1), prefix="bpt", dtype=np.float32)
d_pt = pd.get_dummies(df["d_type"].where(df.has_partd == 1), prefix="dpt", dtype=np.float32)
FEATS = B_FEATS + D_FEATS + FLAGS + list(b_pt.columns) + list(d_pt.columns)
X_all = np.hstack([df[B_FEATS + D_FEATS + FLAGS].to_numpy(np.float32),
                   b_pt.to_numpy(np.float32), d_pt.to_numpy(np.float32)])
y_all = df[LABEL_COL].to_numpy(np.int8)
grp   = df["group"].to_numpy()
ci_ = {f: i for i, f in enumerate(FEATS)}
DS_COLS = {"B":  [ci_[c] for c in B_FEATS + FLAGS + list(b_pt.columns)],
           "D":  [ci_[c] for c in D_FEATS + FLAGS + list(d_pt.columns)],
           "BD": list(range(len(FEATS)))}
del b_pt, d_pt
for k, v in DS_COLS.items(): print(k, "features:", len(v))
print("X_all:", X_all.shape, f"{X_all.nbytes/1e9:.1f} GB")

(7423263, 177) | fraud rows: 2159
B feats: 86 | D feats: 80


B features: 192
D features: 292
BD features: 482
X_all: (7423263, 482) 14.3 GB


In [4]:
npi = df["npi"].to_numpy()
npi_ever = df.groupby("npi")[LABEL_COL].max()

def assign_folds(seed):
    rng = np.random.default_rng(seed)
    parts = []
    for cls in (0, 1):
        ids = rng.permutation(npi_ever.index[npi_ever == cls].to_numpy())
        parts.append(pd.Series(np.arange(len(ids)) % N_FOLDS, index=ids))
    return pd.Series(npi).map(pd.concat(parts)).to_numpy()

f0 = assign_folds(0)
chk = pd.DataFrame({"npi": npi, "fold": f0, "y": y_all})
assert chk.groupby("npi").fold.nunique().max() == 1, "NPI leaks across folds"
print(chk.groupby("fold").agg(rows=("y", "size"), fraud_rows=("y", "sum")))
del chk

         rows  fraud_rows
fold                     
0     1485519         429
1     1484214         421
2     1484881         426
3     1485624         434
4     1483025         449


In [5]:
def evaluate(y_true, p, thr):
    if y_true.min() == y_true.max():
        return None
    pred = (p >= thr).astype(np.int8)
    tp = ((pred == 1) & (y_true == 1)).sum(); fn = ((pred == 0) & (y_true == 1)).sum()
    tn = ((pred == 0) & (y_true == 0)).sum(); fp = ((pred == 1) & (y_true == 0)).sum()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    return {"AUC": roc_auc_score(y_true, p), "AUPRC": average_precision_score(y_true, p),
            "base_rate": y_true.mean(), "TPR": tpr, "TNR": tnr, "GMean": np.sqrt(tpr * tnr),
            "test_pos": int(y_true.sum())}

In [6]:
done = set()
if RES_CSV.exists():
    prev = pd.read_csv(RES_CSV)
    done = set(zip(prev.run, prev.fold, prev.dataset, prev.learner))
print("Already done:", len(done) // 4)

for run in range(N_RUNS):
    folds = assign_folds(run)
    for k in FOLDS_TO_RUN:
        te = np.where(folds == k)[0]; tr = np.where(folds != k)[0]
        thr = y_all[tr].mean()
        for ds, cols in DS_COLS.items():
            for lr in LEARNERS:
                if (run, k, ds, lr) in done: continue
                t0 = time.time()
                m = make_model(lr, seed=run).fit(X_all[np.ix_(tr, cols)], y_all[tr])
                p = m.predict_proba(X_all[np.ix_(te, cols)])[:, 1]
                sec = round(time.time() - t0, 1)
                rows = []
                for g in ["all", "both", "B_only", "D_only"]:
                    sel = slice(None) if g == "all" else (grp[te] == g)
                    r = evaluate(y_all[te][sel], p[sel], thr)
                    if r: rows.append({"run": run, "fold": k, "dataset": ds, "learner": lr,
                                       "label": LABEL_COL, "group": g, **r, "fit_sec": sec})
                pd.DataFrame(rows).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)
                a = {r["group"]: r["AUC"] for r in rows}
                print(f"run{run} fold{k} {ds:>2} {lr}: " +
                      " ".join(f"{g}={v:.4f}" for g, v in a.items()) + f" ({sec}s)")

Already done: 0


run0 fold0  B XGB: all=0.7697 both=0.8305 B_only=0.7845 D_only=0.5000 (68.5s)


run0 fold0  D XGB: all=0.7679 both=0.8127 B_only=0.5000 D_only=0.7893 (101.2s)


run0 fold0 BD XGB: all=0.8145 both=0.8150 B_only=0.8115 D_only=0.7817 (168.6s)


run0 fold1  B XGB: all=0.7499 both=0.8156 B_only=0.7739 D_only=0.5000 (68.4s)


...

run5 fold3  D XGB: all=0.7285 both=0.7530 B_only=0.5000 D_only=0.7855 (100.9s)


run5 fold3 BD XGB: all=0.7990 both=0.8031 B_only=0.7453 D_only=0.8014 (167.4s)


run5 fold4  B XGB: all=0.7659 both=0.8513 B_only=0.7635 D_only=0.5000 (68.0s)


run5 fold4  D XGB: all=0.7344 both=0.8006 B_only=0.5000 D_only=0.7402 (100.9s)


run5 fold4 BD XGB: all=0.8136 both=0.8501 B_only=0.7910 D_only=0.7423 (167.9s)


In [7]:
res = pd.read_csv(RES_CSV)
def ci(s):
    m = s.mean(); h = 1.96 * s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan
    return f"{m:.4f} ({m-h:.4f}, {m+h:.4f})"
for metric in ["AUC", "AUPRC"]:
    for lr in LEARNERS:
        t = (res[res.learner == lr]
             .pivot_table(index="dataset", columns="group", values=metric, aggfunc=ci)
             .reindex(index=["B", "D", "BD"], columns=["all", "both", "B_only", "D_only"]))
        print(f"\n{metric} — {lr}, label={LABEL_COL}, mean (95% CI)\n", t.to_string())
        t.to_csv(RES_DIR / f"{TAG}_{lr}_{metric}.csv")
lift = res.assign(lift=res.AUPRC / res.base_rate).pivot_table(index="dataset", columns="group", values="lift")
print("\nAUPRC / base rate (× better than random)\n", lift.round(2).to_string())


AUC — XGB, label=fraud_nob4, mean (95% CI)
 group                        all                     both                   B_only                   D_only
dataset                                                                                                    
B        0.7495 (0.7445, 0.7545)  0.8091 (0.8014, 0.8168)  0.7697 (0.7591, 0.7804)  0.5000 (0.5000, 0.5000)
D        0.7470 (0.7410, 0.7530)  0.7748 (0.7653, 0.7844)  0.5000 (0.5000, 0.5000)  0.7963 (0.7860, 0.8066)
BD       0.8044 (0.7990, 0.8098)  0.8138 (0.8060, 0.8215)  0.7758 (0.7623, 0.7893)  0.7923 (0.7824, 0.8022)

AUPRC — XGB, label=fraud_nob4, mean (95% CI)
 group                        all                     both                   B_only                   D_only
dataset                                                                                                    
B        0.0023 (0.0019, 0.0027)  0.0033 (0.0028, 0.0038)  0.0070 (0.0041, 0.0100)  0.0004 (0.0004, 0.0004)
D        0.0024 (0.0020, 0.0027)  0.0024 (0.